# Collection

In [ ]:
from glob import glob

from src.common.model import DEFAULT_FULL_DATASET

## SOTA

In [ ]:
from src.sota.model import SOTA, SOTAConstructorArgs, SOTAModelInitializeArgs
from src.sota.evaluate import collect_evaluation_performance as collect_sota

In [ ]:
# Best performing SOTA model
sota_model = SOTA(args=SOTAConstructorArgs(name="yolo11n-full-kf-fold4",
    model_initialize_args=SOTAModelInitializeArgs(model_arch="yolo11n-cls"),
    dataset_name=DEFAULT_FULL_DATASET))
sota_model.initialize_model()

In [ ]:
video_paths = glob("data/videos/**/*.*", recursive=True)
for video_path in video_paths:
    collect_sota(video_path, sota_model)

## HPE DNN

In [ ]:
from src.hpe_dnn.model import HpeDnn, HpeDnnConstructorArgs, HpeDnnModelInitializeArgs
from src.hpe_dnn.architecture import DnnArch
from src.hpe_dnn.evaluate import collect_evaluation_performance as collect_hpednn

In [ ]:
# Best performing DNN model
dnn_model = HpeDnn(args=HpeDnnConstructorArgs(
    name="arch2-no-bal-no-aug-kf-fold9", 
    model_initialize_args=HpeDnnModelInitializeArgs(
        model_arch=DnnArch.ARCH2
    ),
    dataset_name=DEFAULT_FULL_DATASET))
dnn_model.initialize_model()

In [ ]:
dnn_model.model.summary()

In [ ]:
video_paths = glob("data/videos/**/*.*", recursive=True)
for video_path in video_paths:
    collect_hpednn(video_path, dnn_model)

## RNN

In [ ]:
from numpy import load
from os.path import join

from src.common.helpers import read_dataframe
from src.rnn.architecture import RnnArch
from src.rnn.data import WindowGenerator
from src.rnn.model import Rnn, RnnConstructorArgs, RnnModelInitializeArgs
from src.rnn.evaluate import collect_evaluation_performance

In [ ]:
# Best performing RNN model
rnn_model = Rnn(
    args=RnnConstructorArgs(
        name="arch5-medium-dense-fold1",
        model_initialize_args=RnnModelInitializeArgs(
            model_arch=RnnArch.ARCH5,
            input_width=15,
            spacing=1
        )
    )
)
rnn_model.initialize_model()

In [ ]:
rnn_model.model.summary()

In [ ]:
df = read_dataframe("data/df/rnn/cvs_features.pkl")
path = "data/runs/rnn/arch5-medium-dense-fold1/split"
train_split = load(join(path, "train.npy"))
val_split = load(join(path, "val.npy"))
test_split = load(join(path, "test.npy"))

wg = WindowGenerator(df, train_split, val_split, test_split, 
    rnn_model.model_initialize_args.input_width, rnn_model.model_initialize_args.spacing)

In [ ]:
video_paths = glob("data/videos/**/*.*", recursive=True)

for video_path in video_paths:
    collect_evaluation_performance(video_path, rnn_model, wg)

## CONV LSTM

In [ ]:
from glob import glob
from numpy import load
from os.path import join

from src.common.helpers import read_dataframe
from src.conv_lstm.architecture import ConvLstmArch
from src.conv_lstm.data import WindowGenerator
from src.conv_lstm.model import ConvLstm, ConvLstmConstructorArgs, ConvLstmModelInitializeArgs
from src.conv_lstm.evaluate import collect_evaluation_performance

In [ ]:
# Best performing RNN model
conv_lstm_model = ConvLstm(
    args=ConvLstmConstructorArgs(
        name="arch1-small-fold3",
        model_initialize_args=ConvLstmModelInitializeArgs(
            model_arch=ConvLstmArch.ARCH1,
            input_width=5,
            spacing=1
        )
    )
)
conv_lstm_model.initialize_model()

In [ ]:
conv_lstm_model.model.summary()

In [ ]:
df = read_dataframe("data/df/rnn/cvs_features.pkl")
path = "data/runs/conv_lstm/arch1-small-fold3/split"
train_split = load(join(path, "train.npy"))
val_split = load(join(path, "val.npy"))
test_split = load(join(path, "test.npy"))

wg = WindowGenerator(df, train_split, val_split, test_split, 
    conv_lstm_model.model_initialize_args.input_width, 
    conv_lstm_model.model_initialize_args.spacing)

In [ ]:
video_paths = glob("data/videos/**/*.*", recursive=True)

for video_path in video_paths:
    collect_evaluation_performance(video_path, conv_lstm_model, wg)

# Print results

In [ ]:
from src.sota.evaluate import print_results as print_sota_results
from src.hpe_dnn.evaluate import print_results as print_dnn_results
from src.rnn.evaluate import print_results as print_rnn_results
from src.conv_lstm.evaluate import print_results as print_conv_lstm_results

In [ ]:
evaluation_root = "data/df/evaluation_results"

print_sota_results(evaluation_root)
print_dnn_results(evaluation_root)
print_rnn_results(evaluation_root)
print_conv_lstm_results(evaluation_root)